In [1]:
%pip install pandas sqlalchemy pymysql

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 26.8 MB/s  0:00:00

   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [pymysql]
   ---------------------------------------- 2/2 [pymysql]

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

os.environ["DISABLE_SQLALCHEMY_CEXT"] = "1"

In [2]:
%pip install --force-reinstall --no-cache-dir --no-binary=SQLAlchemy SQLAlchemy

     ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
     --------- ------------------------------ 2.6/10.5 MB 22.1 MB/s eta 0:00:01
     ---------------------- ----------------- 6.0/10.5 MB 18.5 MB/s eta 0:00:01
     ---------------------------------------- 10.5/10.5 MB 22.2 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for SQLAlchemy: filename=sqlalchemy-2.1.4-py3-none-any.whl size=2054965 sha256=65980a3d3001f62c82dd962f782a21b72a34d37a4bf8391039234928ec7f782f
  Stored in directory: C:\Users\julia\AppData\Local\Temp\pip-ephem-wheel-cache-amjbn3bs\wheels\9b\ac\e5\12e14f9a3a88c0aebcf62b2c0b9d34fb19955b2bfc0d1e42f3
Successfully built SQLAlchemy

  Attempting u

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text, URL
from getpass import getpass

In [2]:
password = getpass("Introduz a password do MySQL: ")

url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    port=3306,
    database="sakila"
)

engine = create_engine(url)

In [3]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print("Base de dados ligada:", result.scalar())

Base de dados ligada: sakila


2. SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, and return it as a pandas DataFrame

In [13]:
def rentals_month(engine, month, year):
    query = text("""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = :month
          AND YEAR(rental_date) = :year
    """)

    with engine.connect() as connection:
        df = pd.read_sql(
            query,
            connection,
            params={"month": month, "year": year}
        )

    return df


rentals_may = rentals_month(engine, 5, 2005)
rentals_june = rentals_month(engine, 6, 2005)

print("May:", rentals_may.shape)
print("June:", rentals_june.shape)

May: (1156, 7)
June: (2311, 7)


3. Develop a Python function called rental_count_month that takes the DataFrame provided by rentals_month as input along with the month and year and returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year.

The function should also include the month and year as parameters and use them to name the new column according to the month and year, for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".

Hint: Consider making use of pandas groupby()

In [17]:
def rental_count_month(df, month, year):
    column_name = f"rentals_{month:02d}_{year}"

    counts = (df.groupby("customer_id")
        .size()
        .reset_index(name=column_name))

    return counts


count_may = rental_count_month(rentals_may, 5, 2005)
count_june = rental_count_month(rentals_june, 6, 2005)

display(count_may.head())
display(count_june.head())

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


4. Create a Python function called compare_rentals that takes two DataFrames as input containing the number of rentals made by each customer in different months and years. The function should return a combined DataFrame with a new 'difference' column, which is the difference between the number of rentals in the two months.

In [18]:
def compare_rentals(df1, df2):
    column_month1 = df1.columns[1]
    column_month2 = df2.columns[1]

    comparison = pd.merge(df1, df2, 
        on="customer_id",
        how="inner")

    comparison["difference"] = (comparison[column_month2] - comparison[column_month1])
    
    return comparison


comparison = compare_rentals(count_may, count_june)

display(comparison.head())

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1
